<a href="https://colab.research.google.com/github/zbovaird/Agentic-Hemispheres/blob/dev/Cursor_Architecture_Update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cursor AI Summary Architecture

Use this document as the design spec for adding summary-first AI navigation and schema-safe inter-agent communication to the Cursor bicameral template.

The goal is not to comment every block of code. The goal is to make the project easier for the Master and Emissary to navigate by creating and maintaining a small number of durable, high-signal summaries in the places that materially reduce exploration cost.

## Outcome

The template should make Cursor behave as if summary maintenance is part of normal architecture hygiene:

- At project bootstrap, the repo starts with a summary map instead of waiting until the codebase is already hard to navigate.
- During implementation, the Master and Emissary refresh nearby summaries when new domains, modules, or large files appear.
- During review, summary drift is treated as architecture drift.
- During refactors, summaries are consolidated, moved, or deleted along with the code they describe.

## Core Principle

Use summary-first routing, not comment saturation.

That means:

- Prefer a repo map, folder-level README files, and targeted file-orientation comments.
- Add summaries only where they reduce search cost or prevent repeated rediscovery.
- Do not add repetitive comments to every block.
- Do not add comments to native JSON files.
- Do not force summaries into tiny files that are already cheaper to read directly.

## What The Architecture Should Produce

The template should maintain four summary layers.

### Layer 1: Repo Map

Purpose: give the Master a cheap first-pass map of the repository.

Typical artifacts:

- `README.md`
- `docs/repository-structure.md`
- `docs/architecture-map.md` or equivalent

This layer answers:

- what the repo is for
- where the major domains live
- which folders are expensive to inspect
- where to start for onboarding

### Layer 2: Domain Or Folder Summaries

Purpose: prevent agents from opening every file in large or mixed-purpose folders.

Typical artifacts:

- `README.md` inside important folders
- one short architecture note per major domain when a README is not appropriate

This layer answers:

- what this area owns
- which files matter most
- what patterns or invariants are local to this area
- when the raw source must still be opened

### Layer 3: Large File Orientation

Purpose: make expensive files searchable without reading them top to bottom.

Typical artifacts:

- a short orientation comment at the top of a large comment-capable file
- a sidecar summary file if the source format is not comment-friendly or the file is generated/exported

This layer answers:

- why the file exists
- the highest-value entry points
- local invariants and sharp edges
- which sections are likely to matter during debugging or implementation

### Layer 4: Shared State Summaries

Purpose: keep task history compressed so the Master does not need full replay.

Typical artifacts:

- `.cursor/plans/workspace_state.json`
- durable planning notes under `.cursor/plans/`

This layer answers:

- current constraints
- known surprises
- summary debt or missing summaries to add later

## Placement Rules

Add or refresh summaries only when one of these conditions is true.

### Add A Repo Or Domain Summary When

- a new top-level domain is created
- a folder mixes multiple concerns and agents repeatedly reopen the same files to rediscover structure
- a folder exceeds roughly 8 to 12 meaningful files
- a folder contains more than one implementation language or more than one workflow surface

### Add A Large-File Summary When

- a file exceeds roughly 300 to 500 lines
- a file exceeds roughly 12 KB to 20 KB
- a file is repeatedly used as reference material
- the file is an export, generated artifact, rule pack, or dense configuration surface
- the file cannot be understood cheaply from its name alone

### Do Not Add A Summary When

- the file is tiny and cheaper to read than to summarize
- the summary would just restate the code line by line
- the file is native JSON and the summary would have to live inside the JSON itself
- the file is transient or scratch material with no stable architectural role

## Summary Content Contract

Every summary should compress navigation cost, not paraphrase everything.

Good summaries usually contain:

- purpose
- main responsibilities
- important entry points or actions
- inputs and outputs when relevant
- current invariants or constraints
- related files
- when to read the raw source anyway

Avoid:

- line-by-line narration
- stale counts or low-value detail that drifts quickly
- full code duplication
- generic text that could describe any module

## Lifecycle Policy

This feature only works if it is part of the project lifecycle, not a cleanup pass months later.

### Phase 1: Project Bootstrap

From the start of a new project, the template should create:

- a repo-level map
- a repository-structure note
- a summary policy note or rule
- starter README files for any known heavyweight folders

At bootstrap, the Master should assume summary coverage is part of the project skeleton, just like tests, rules, and workspace state.

### Phase 2: Imported Or Existing Repo Onboarding

When the harness is dropped into an existing repository:

- the Master or onboarding Emissary should identify the major domains first
- create or refresh repo and folder summaries before deeper implementation
- record summary gaps in `.cursor/plans/workspace_state.json`

For imported repos, the first summary pass should happen before normal coding work begins.

### Phase 3: Feature Development

During normal implementation:

- if a feature creates a new architectural area, add the nearest summary in the same change
- if a touched file becomes large enough to be expensive, add a brief orientation comment or sidecar summary
- if a summary becomes wrong because behavior moved, refresh it in the same task

The Master should include summary maintenance in acceptance criteria whenever the shape of the codebase changes materially.

### Phase 4: Review And Approval

During review:

- if the code changed enough that the existing summary is misleading, treat that as review debt
- if a large new file was added without any summary surface, ask whether it crossed the threshold for summary coverage
- if summaries are stale, the Master should either require an update or record explicit follow-up debt

### Phase 5: Refactor And Consolidation

During refactors:

- merge summaries when modules consolidate
- move summaries when ownership moves
- delete summaries that no longer describe a stable surface
- reduce summary sprawl if the codebase becomes simpler

## Bicameral Responsibilities

This feature works best when each hemisphere owns a different part of the policy.

### Master Responsibilities

- use summaries first during initial exploration
- decide where summary coverage is needed
- include summary maintenance in the handshake when architectural shape changes
- reject noisy or redundant summary creation
- track summary debt in `.cursor/plans/workspace_state.json`

### Emissary Responsibilities

- update the nearest relevant summary when touching a file that changes structure or invariants
- create a new summary surface when the Master's acceptance criteria require it
- keep summaries compact and local to the owning domain
- avoid summary sprawl and avoid editing unrelated summaries

## Automation Policy

To make this feel automatic in Cursor, the template should combine rules, conventions, and optional hooks.

### Required Automation: Rule-Level Behavior

The Cursor rules should instruct the agents to:

- read summaries before raw heavy files on first pass
- add or refresh summaries when codebase shape changes
- avoid block-level comment spam
- prefer sidecar summaries for exported or generated files

This is the minimum viable architecture and should exist in the template from day one.

### Recommended Automation: Review-Level Enforcement

The Master should treat missing or stale summaries as a review consideration when:

- a large file is introduced
- a new folder or domain is added
- a module's responsibility changes materially

This does not require a hard blocker on every task. It requires architectural awareness.

### Optional Automation: Hook Or Script Support

If you want stronger enforcement later, add a lightweight script or hook that flags likely summary debt when:

- a newly created file exceeds the large-file threshold
- a new folder has no summary surface
- a changed summary no longer mentions a renamed major file

This should warn or annotate first. It should not hard-fail early development unless the team explicitly wants that.

## Suggested Cursor Template Changes

Use the architecture below in the template.

### 1. Add A Cursor Rule

Recommended file:

- `.cursor/rules/05_ai_summaries.mdc`

Ready-to-paste content:

```md
---
description: USE WHEN exploring the repo, adding new modules, creating large files, or changing folder structure.
globs:
alwaysApply: true
---

# AI Summary Lifecycle

<!-- Summary rule: use summary-first routing to reduce exploration cost. Prefer repo maps, folder READMEs, and brief orientation comments over exhaustive inline commentary. -->

Use summaries as a first-pass navigation layer.

## Summary-First Exploration

- Before opening a large file or mixed-purpose folder, look for the nearest README, architecture note, or orientation summary.
- If a summary answers the routing question, do not read the full source yet.
- Open raw source only when implementation or debugging requires exact details.

## When To Create Or Refresh Summaries

- A new top-level or domain folder is added.
- A file becomes large, dense, exported, or repeatedly referenced.
- A module's responsibility or boundary changes materially.
- A nearby summary becomes stale because of the current change.

## Preferred Summary Surfaces

- Repo map documents in `README.md` and `docs/`.
- Folder-level `README.md` files for major domains.
- Brief orientation comments at the top of large comment-capable files.
- Sidecar summaries for exported, generated, or comment-hostile files.

## Avoid

- Commenting every block.
- Writing summaries for tiny files.
- Duplicating code behavior line by line.
- Adding comments inside native JSON.

## Bicameral Policy

- The Master decides where summary coverage is architecturally useful.
- The Emissary updates the nearest relevant summary when implementing structural changes.
- Summary debt should be tracked in `.cursor/plans/workspace_state.json` when not fixed in the current task.
```

### 2. Add A Repo-Level Summary Policy Note

Recommended file:

- `docs/summary-policy.md`

Purpose:

- gives the repo a stable explanation of where summaries belong
- provides thresholds and anti-patterns
- becomes a reusable reference for both Master and Emissary

### 3. Add Summary Coverage To Bootstrap

During template bootstrap, create:

- `README.md`
- `docs/repository-structure.md`
- `docs/summary-policy.md`
- starter READMEs in any heavyweight default folders the template ships with

### 4. Track Summary Debt In Workspace State

When summary work is deferred, store items such as:

```json
{
  "summary_debt": [
    "Add folder README for infra/connectors after current feature lands",
    "Refresh src/agents/README.md after prompt router refactor"
  ]
}
```

### 5. Preserve Canonical JSON Transport

Recommended files:

- `.cursor/rules/01_master_rh.mdc`
- `.cursor/rules/03_callosum.mdc`
- `.cursor/plans/workspace_state.json`

Purpose:

- keeps the Master free to use model-optimized prompt structure internally
- keeps the Emissary on strict machine-readable outputs
- prevents communication breakdown by enforcing one shared runtime transport format

## Recommended Summary Shapes

### Folder README Shape

- what this folder owns
- which files matter most
- local patterns or invariants
- when to read raw files

### Large File Orientation Shape

- why this file exists
- key entry points
- major sections
- sharp edges or constraints

### Repo Map Shape

- major domains
- agent/rule locations
- expensive surfaces
- preferred onboarding path

## Schema Strategy

The same architectural principle used for summaries should apply to inter-agent communication: optimize prompts for the model that consumes them, but keep the transport format canonical and deterministic.

**Schema Rule:** All inter-agent communication and shared state use JSON-only transport; prompt layers are model-specific and optimized for the target model.

### Core Decision

Use mixed prompt formatting with canonical JSON transport.

That means:

- Claude Opus 4.6 may use XML internally for prompt organization.
- Gemini 2.5 Flash should return strict JSON for machine-facing outputs.
- The Master and Emissary should exchange JSON only across the corpus callosum boundary.
- `.cursor/plans/workspace_state.json` remains the canonical shared-state format.

### Format Rules By Context

#### Master Internal Prompt Format

XML is a good fit for the Master when you want clear separation of:

- goals
- constraints
- review criteria
- handoff expectations
- summary policy

Use XML only as an internal prompt-structuring tool for the Master. Do not use XML as the runtime transport format between agents.

#### Emissary Output Format

Gemini 2.5 Flash should be constrained to strict JSON for outputs that the Master or Cursor automation must parse.

Good uses:

- handshakes
- `implementation_proof`
- `SURPRISE`
- `ESCALATE`
- review-ready summaries with fixed keys

This keeps the Emissary fast and machine-readable.

#### Cross-Agent Transport Format

The callosum transport should stay JSON only.

That includes:

- intent handshakes
- signal payloads
- implementation proof payloads
- workspace state
- optional validation artifacts

This is the contract that must not drift.

### Why JSON Is The Best Emissary Transport

JSON is the strongest choice for Gemini 2.5 Flash in Cursor because it is:

- easy to validate
- unambiguous to parse
- strong for nested task and result payloads
- already aligned with `.cursor/plans/workspace_state.json`
- a natural fit for deterministic hooks, scripts, and review logic

A tiny example:

```json
{
  "schema_version": "1.0",
  "intent_id": "add-swimlane-api-auth",
  "architectural_constraint": "MCP stdio transport only; no external services",
  "target_files": ["src/swimlane-client.ts", "tests/swimlane-client.test.ts"],
  "acceptance_criteria": ["PAT auth support added", "tests pass", "no breaking changes"]
}
```

### Why YAML Should Not Be The Runtime Handoff Format

YAML is acceptable for human-authored configuration, but it is a weaker runtime transport for agent handoffs.

Use YAML for:

- static repository configuration
- editor or CI config
- human-maintained manifests

Do not use YAML for:

- Master to Emissary handshakes
- Emissary return payloads
- signal transport
- shared runtime state

YAML tolerates indentation-sensitive ambiguity that makes it fragile for runtime handoffs where agents programmatically generate and parse payloads, while JSON uses explicit delimiters and produces more deterministic parsing.

### Why TOML Is Not Recommended

TOML is fine for flat configuration files, but it is not the best fit for bicameral runtime payloads.

Avoid TOML for cross-agent transport because:

- it is weaker for nested arrays and objects
- it is less natural for task and result structures
- it adds no practical advantage over JSON for Cursor automation
- it is not the strongest output target for Gemini Flash

### Guardrails For Reliability

To prevent communication breakdown between Opus and Gemini:

- include `schema_version` at the payload root
- use fixed top-level keys
- avoid dynamic keys when a stable field name will do
- forbid extra prose outside the JSON payload
- keep return types deterministic
- validate required keys before Master review when automation is available

### Why This Fits The Summary Architecture

This summary-first architecture is already built around compression, routing, and canonical state.

The schema strategy should follow the same pattern:

- model-specific formatting is allowed inside each hemisphere
- shared transport must stay boring and uniform
- the Master decides what the structure should mean
- the Emissary returns the smallest reliable machine-readable payload

This keeps the architecture coherent: native prompts, unified transport, and low-friction review.

### Final Schema Design Decision

Use XML for Master-side prompt organization when it improves reasoning clarity.

Use JSON for Emissary outputs and all cross-agent transport.

Use YAML for static human-authored configuration only.

Do not use TOML as the primary interchange format for Master and Emissary communication.

## Example Trigger Matrix

| Event | Expected Summary Action |
|---|---|
| New project bootstrap | Create repo map and default summary surfaces |
| Imported repo onboarding | Create first-pass repo and folder summaries |
| New domain folder | Add folder README |
| Large exported file added | Add sidecar summary or top-level orientation note |
| Major refactor | Refresh related summaries in same change |
| Review finds stale summary | Fix immediately or record summary debt |

## Anti-Patterns

Do not implement this feature as:

- exhaustive block comments across the codebase
- one giant summary file for the entire repo
- summaries that are never refreshed
- generated summaries with no ownership model
- hard-fail hooks on every tiny change

## Acceptance Criteria For The Template

The feature is working correctly when:

- a new repo starts with a usable summary-first navigation layer
- the Master reaches for summaries before raw heavy files
- the Emissary updates nearby summaries when code shape changes
- large exported or dense files gain sidecar summaries instead of inline noise
- summary debt is visible and intentionally managed

## Recommended Rollout

### Minimum Viable Version

- add the Cursor rule
- add `docs/summary-policy.md`
- ensure bootstrap creates repo and folder summary scaffolding

### Stronger Version

- add a lightweight check that flags missing summary surfaces on large new files or domains
- track summary debt in workspace state
- add imported-repo onboarding behavior that creates first-pass summaries before normal coding

### Full Version

- connect repo discovery, bootstrap, and review policy so summaries are maintained as part of normal architectural governance
- use hooks only to flag likely debt, not to replace architectural judgment

## Final Design Decision

The architecture should optimize for durable navigation, not total documentation coverage.

If a summary does not change how an agent chooses where to look next, it should probably not exist.